# Forest Segmentation — SegFormer-B0 Baseline (Colab)

Week 1-2 goal: train a plain SegFormer-B0 (no attention loss yet) on the full
dataset and report Dice / IoU. Every piece here was already verified locally
in VS Code on a small subset — this notebook just scales it up with a real
GPU and the full dataset.

**Before running:** make sure Runtime → Change runtime type → GPU (T4) is selected.

## Step 0: Mount Google Drive

Upload your `Kalana` folder (with `images/` and `masks/` subfolders) to Google
Drive first, then mount it here. Adjust `DATA_ROOT` below to match wherever
you placed it in your Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers accelerate

## Step 1: Config + verify filename pairing

In [3]:
import os
from pathlib import Path

# ── Config ────────────────────────────────────────────────
# Adjust this path to wherever "Kalana" ended up inside your Drive.
# Example: "/content/drive/MyDrive/Kalana"
DATA_ROOT = Path("/content/drive/MyDrive/Kalana")
IMAGES_DIR = DATA_ROOT / "images"
MASKS_DIR = DATA_ROOT / "masks"

IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 20
LR = 6e-5
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
SEED = 42
CHECKPOINT_OUT = "/content/drive/MyDrive/segformer_b0_baseline.pt"
# ──────────────────────────────────────────────────────────

image_files = sorted(os.listdir(IMAGES_DIR))
mask_files = sorted(os.listdir(MASKS_DIR))

print(f"Found {len(image_files)} images")
print(f"Found {len(mask_files)} masks")

Found 5108 images
Found 5108 masks


In [4]:
def mask_name_to_image_name(mask_filename):
    """Converts '855_mask_01.jpg' -> '855_sat_01.jpg'."""
    return mask_filename.replace("_mask", "_sat")

# Full-dataset pairing check before we trust it on everything
missing = []
for m in mask_files:
    expected_image = mask_name_to_image_name(m)
    if expected_image not in image_files:
        missing.append((m, expected_image))

print(f"Total masks: {len(mask_files)}")
print(f"Missing matches: {len(missing)}")
if missing:
    print("First few missing pairs:", missing[:5])
assert len(missing) == 0, "Fix missing pairs before continuing."

Total masks: 5108
Missing matches: 0


## Step 2: Dataset class

Same logic verified locally — resize, normalize, binarize the mask.

In [5]:
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset

class ForestSegDataset(Dataset):
    def __init__(self, mask_filenames, images_dir, masks_dir, img_size=256):
        self.mask_filenames = mask_filenames
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.img_size = img_size

        # ImageNet normalization stats (what SegFormer's pretrained backbone expects)
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def __len__(self):
        return len(self.mask_filenames)

    def __getitem__(self, idx):
        mask_fname = self.mask_filenames[idx]
        image_fname = mask_name_to_image_name(mask_fname)

        image = Image.open(self.images_dir / image_fname).convert("RGB")
        image = image.resize((self.img_size, self.img_size))

        mask = Image.open(self.masks_dir / mask_fname).convert("L")
        mask = mask.resize((self.img_size, self.img_size), resample=Image.NEAREST)

        image = np.array(image, dtype=np.float32) / 255.0
        image = (image - self.mean) / self.std
        image = torch.from_numpy(image).permute(2, 0, 1).float()

        # White (>127) = forest = 1, black = non-forest = 0
        mask = np.array(mask, dtype=np.int64)
        mask = (mask > 127).astype(np.int64)
        mask = torch.from_numpy(mask).long()

        return image, mask

## Step 3: Train / val / test split

In [6]:
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def make_splits():
    all_files = sorted(mask_files)
    random.shuffle(all_files)

    n = len(all_files)
    n_val = int(n * VAL_SPLIT)
    n_test = int(n * TEST_SPLIT)

    val_files = all_files[:n_val]
    test_files = all_files[n_val:n_val + n_test]
    train_files = all_files[n_val + n_test:]
    return train_files, val_files, test_files

set_seed(SEED)
train_files, val_files, test_files = make_splits()
print(f"Train/Val/Test sizes: {len(train_files)}/{len(val_files)}/{len(test_files)}")

Train/Val/Test sizes: 3576/766/766


## Step 4: Model + Dice/IoU metric

Same architecture verified locally: pretrained MiT-B0 encoder + fresh 2-class decode head.

In [7]:
from transformers import SegformerForSemanticSegmentation
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")

def build_model():
    model = SegformerForSemanticSegmentation.from_pretrained(
        "nvidia/mit-b0",
        num_labels=2,
        id2label={0: "non_forest", 1: "forest"},
        label2id={"non_forest": 0, "forest": 1},
    )
    return model.to(device)

def dice_iou_score(pred_mask, true_mask, eps=1e-6):
    """pred_mask, true_mask: bool tensors, same shape. Dice/IoU for the
    'forest' (positive) class."""
    intersection = (pred_mask & true_mask).sum().float()
    pred_sum = pred_mask.sum().float()
    true_sum = true_mask.sum().float()
    union = pred_sum + true_sum - intersection

    dice = (2 * intersection + eps) / (pred_sum + true_sum + eps)
    iou = (intersection + eps) / (union + eps)
    return dice.item(), iou.item()

Running on: cuda


## Step 5: DataLoaders + training loop

This is the same loop verified locally on 10 images/5 epochs — now scaled up to the full dataset and 20 epochs on GPU.

In [8]:
from torch.utils.data import DataLoader
from tqdm import tqdm

train_ds = ForestSegDataset(train_files, IMAGES_DIR, MASKS_DIR, IMG_SIZE)
val_ds = ForestSegDataset(val_files, IMAGES_DIR, MASKS_DIR, IMG_SIZE)
test_ds = ForestSegDataset(test_files, IMAGES_DIR, MASKS_DIR, IMG_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model = build_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

config.json:   0%|          | 0.00/70.0k [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 14.4MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.weight                                       | UNEXPECTED | 
classifier.bias                                         | UNEXPECTED | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.linear_fuse.weight                          | MISSING    | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.classifier.bias                             | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.batch_norm.weight                           

In [9]:
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_dice, total_iou, n_batches = 0.0, 0.0, 0.0, 0

    with torch.set_grad_enabled(is_train):
        for images, masks in tqdm(loader, leave=False):
            images, masks = images.to(device), masks.to(device)

            outputs = model(pixel_values=images)
            logits = outputs.logits
            logits = F.interpolate(
                logits, size=masks.shape[-2:], mode="bilinear", align_corners=False
            )

            loss = F.cross_entropy(logits, masks)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            preds = logits.argmax(dim=1).bool()
            dice, iou = dice_iou_score(preds, masks.bool())

            total_loss += loss.item()
            total_dice += dice
            total_iou += iou
            n_batches += 1

    return total_loss / n_batches, total_dice / n_batches, total_iou / n_batches

In [10]:
best_val_dice = 0.0
for epoch in range(1, EPOCHS + 1):
    train_loss, train_dice, train_iou = run_epoch(model, train_loader, optimizer)
    val_loss, val_dice, val_iou = run_epoch(model, val_loader)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} dice={train_dice:.4f} iou={train_iou:.4f} | "
        f"val_loss={val_loss:.4f} dice={val_dice:.4f} iou={val_iou:.4f}"
    )

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), CHECKPOINT_OUT)
        print(f"  -> saved new best checkpoint ({CHECKPOINT_OUT})")

Epoch 01 | train_loss=0.4506 dice=0.8341 iou=0.7217 | val_loss=0.3730 dice=0.8643 iou=0.7652
  -> saved new best checkpoint (/content/drive/MyDrive/segformer_b0_baseline.pt)


Epoch 02 | train_loss=0.3974 dice=0.8582 iou=0.7566 | val_loss=0.3736 dice=0.8599 iou=0.7589


Epoch 03 | train_loss=0.3700 dice=0.8693 iou=0.7738 | val_loss=0.3587 dice=0.8739 iou=0.7806
  -> saved new best checkpoint (/content/drive/MyDrive/segformer_b0_baseline.pt)


Epoch 04 | train_loss=0.3411 dice=0.8808 iou=0.7910 | val_loss=0.3882 dice=0.8528 iou=0.7486


Epoch 05 | train_loss=0.3145 dice=0.8914 iou=0.8072 | val_loss=0.3656 dice=0.8745 iou=0.7819
  -> saved new best checkpoint (/content/drive/MyDrive/segformer_b0_baseline.pt)


...

Epoch 16 | train_loss=0.1384 dice=0.9541 iou=0.9129 | val_loss=0.5422 dice=0.8739 iou=0.7803


Epoch 17 | train_loss=0.1310 dice=0.9571 iou=0.9183 | val_loss=0.5343 dice=0.8684 iou=0.7719


Epoch 18 | train_loss=0.1265 dice=0.9586 iou=0.9210 | val_loss=0.5721 dice=0.8725 iou=0.7782


Epoch 19 | train_loss=0.1242 dice=0.9588 iou=0.9215 | val_loss=0.5642 dice=0.8692 iou=0.7734


Epoch 20 | train_loss=0.1172 dice=0.9608 iou=0.9251 | val_loss=0.5939 dice=0.8651 iou=0.7674


## Step 6: Final test-set evaluation

This Dice/IoU number is your Week 1-2 deliverable to report to the team.

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_OUT))
test_loss, test_dice, test_iou = run_epoch(model, test_loader)
print(f"\nFINAL TEST RESULTS: dice={test_dice:.4f}  iou={test_iou:.4f}")